# 04 — Treino com HDF5 nativo (sem decodificação de imagem)

Terceira variante do pipeline de dados, depois do vídeo bruto
(`01_treino_libero.ipynb`) e dos frames pré-processados em JPEG
(`03_treino_rapido_preprocessado.ipynb`). Lê os arrays `uint8` crus do
HDF5 oficial do LIBERO (`128×128`, sem decodificação de imagem nenhuma --
nem vídeo, nem JPEG). Ver `docs/hdf5_migration.md` para o raciocínio
completo: benchmarks de I/O, mapeamento das 40 tarefas, verificação de
convenção de ação e o achado do flip vertical de imagem.

**Diferenças importantes em relação ao `03`:**
- **Resolução nativa 128×128**, não 256×256 -- decisão tomada
  explicitamente (ver `docs/hdf5_migration.md`): menos detalhe visual
  entrando no modelo, então as curvas de loss NÃO são diretamente
  comparáveis com um checkpoint treinado no pipeline JPEG (256×256) do
  mesmo `--config`.
- Por isso este notebook salva os checkpoints numa pasta **separada**
  (sufixo `_hdf5128`) -- nunca reusa/sobrescreve o checkpoint do pipeline
  JPEG do mesmo `--config`.
- **Não precisa de `lerobot[libero]`** (dependência pesada). O import do
  lerobot em `scripts/train.py` agora é local (lazy), só acontece no
  caminho de vídeo bruto (sem `--preprocessed-dir`/`--hdf5-dir`). Ainda
  precisa de `sentence-transformers` (extra `language`) se o `--config`
  usar fusão de linguagem (film/token/cross_attn), como `language_40_film`
  abaixo.
- O dataset (`.hdf5`) vai pro disco **local** do Colab (`/content`), não
  pro Drive -- são ~700-800MB por tarefa, e ler via Drive FUSE é lento
  (o oposto do que a migração busca). É baixado de novo a cada sessão
  nova do Colab (retomável: pula o que já baixou, se a sessão cair no
  meio). Só os **checkpoints** (pequenos) vão pro Drive.

**Piloto antes do treino completo**: baixar as 40 tarefas dá ~28-32GB --
comece com `PILOT_LIMIT` pequeno (poucas tarefas) pra validar que tudo
roda de ponta a ponta antes de comprometer o download completo e horas
de GPU.

## 1. Repositório e ambiente

In [ ]:
!git clone -b correcoes-set2026 https://github.com/rafaelheydt/act-lang.git
%cd act-lang

# "language" traz sentence-transformers -- necessário pros mecanismos de fusão
# (film/token/cross_attn); "hdf5" traz h5py + huggingface_hub. Ainda sem
# lerobot[libero] (pesado, só necessário no caminho de vídeo bruto).
!pip install -q -e ".[hdf5,language]"

## 2. Baixar o dataset HDF5 (disco local do Colab)

`PILOT_LIMIT`: `None` baixa as 40 tarefas (~28-32GB); um número baixa só
as N primeiras (ordem alfabética, determinística -- ver
`scripts/download_libero_hdf5.py`). Comece pequeno.

In [ ]:
PILOT_LIMIT = 3  # None = as 40 tarefas completas (~28-32GB); comece pequeno pra validar
DATA_DIR = "/content/libero_hdf5"

import subprocess, sys

cmd = [sys.executable, "-u", "scripts/download_libero_hdf5.py", "--out", DATA_DIR]
if PILOT_LIMIT is not None:
    cmd += ["--limit", str(PILOT_LIMIT)]
print("rodando:", " ".join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\ncódigo de saída:", proc.returncode)
assert proc.returncode == 0, "download falhou -- veja o log acima"

## 3. Config: ajustar `device_index` para o Colab

`configs/libero_40tasks_language.py` tem `device_index: 1`, escolhido para
a máquina do CEPEDI (2 GPUs -- 0=A2000, 1=RTX 3050). No Colab normalmente
só existe 1 GPU (índice 0); deixamos `None` para o `pick_device` escolher
sozinho, funcionando em qualquer ambiente com 1+ GPUs.

In [ ]:
!sed -i 's/"device_index": 1,/"device_index": None,  # None = auto -- era 1 p\/ CEPEDI (2 GPUs)/' configs/libero_40tasks_language.py
!grep -n "device_index" configs/libero_40tasks_language.py

## 4. Treino

Monta o Drive só para persistir os **checkpoints** (pequenos -- não é o
dataset, que fica em `DATA_DIR` no disco local). `CHECKPOINT_DIR` usa o
sufixo `_hdf5128` -- nunca colide com um checkpoint do mesmo `--config`
treinado no pipeline JPEG 256×256 (ver nota no topo do notebook: os dois
não são diretamente comparáveis/retomáveis um a partir do outro).

Escolha o mecanismo trocando `CONFIG` abaixo: `language_40_film` |
`language_40_token` | `language_40_cross_attn` (nomes exatos em
`CONFIG_REGISTRY`, em `scripts/train.py`) -- e ajuste o import de `cfg` na
célula de curvas (seção 5) para o mesmo mecanismo.

In [ ]:
CONFIG = "language_40_film"

from google.colab import drive
drive.mount('/content/drive')

import sys
from configs.libero_40tasks_language import CONFIG_FILM as cfg  # troque se mudou CONFIG acima
CHECKPOINT_DIR = f"/content/drive/MyDrive/{cfg['experiment_name']}_hdf5128"
print("checkpoints em:", CHECKPOINT_DIR)

import subprocess
from datetime import datetime
from pathlib import Path

Path("logs").mkdir(exist_ok=True)
log_path = f"logs/treino_hdf5_{datetime.now():%Y%m%d_%H%M}.log"

cmd = [
    sys.executable, "-u", "scripts/train.py",
    "--config", CONFIG,
    "--hdf5-dir", DATA_DIR,
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--resume",
]
print("rodando:", " ".join(cmd))
print("log em:", log_path, "\n" + "=" * 60)

with open(log_path, "a") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

print("\n" + "=" * 60)
print(f"processo encerrado com código {proc.returncode}"
      + (" -- verifique o log acima" if proc.returncode != 0 else " -- ok"))

## 5. Curvas

Carrega o `history` salvo no `last_checkpoint.pt` de `CHECKPOINT_DIR`
(ele viaja junto com o checkpoint -- não precisa recomputar nada).

In [ ]:
import torch
from act_lang.training.checkpoints import load_checkpoint
from configs.libero_40tasks_language import CONFIG_FILM as cfg  # troque se usou outro mecanismo
from act_lang.models.act import ACT
from act_lang.models.fusion import build_fusion

model = ACT(
    action_dim=cfg["action_dim"], state_dim=cfg["state_dim"], d_model=cfg["d_model"],
    latent_dim=cfg["latent_dim"], chunk_size=cfg["chunk_size"], n_cameras=cfg["n_cameras"],
    n_encoder_layers=cfg["n_encoder_layers"], n_decoder_layers=cfg["n_decoder_layers"],
    n_heads=cfg["n_heads"], dropout=cfg["dropout"], decoder_style=cfg["decoder_style"],
    fusion=build_fusion(cfg["fusion_type"], cfg["d_model"]),
)
from pathlib import Path
_, history = load_checkpoint(Path(CHECKPOINT_DIR) / "last_checkpoint.pt", model, device="cpu")

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss total (recon + kl_weight*KL)")

axes[1].plot(history["train_recon"], label="train (z~q)")
axes[1].plot(history["val_recon"], label="val (z=mu)")
axes[1].plot(history["val_recon_z0"], label="val (z=0)", linestyle="--")
axes[1].set_title("Recon L1 — z0 é a métrica de seleção")

axes[2].plot(history["train_kld"], label="train")
axes[2].plot(history["val_kld"], label="val")
axes[2].set_title("kld_raw")

axes[3].plot(history["train_mu_abs_mean"], label="train")
axes[3].plot(history["val_mu_abs_mean"], label="val")
axes[3].axhline(0, color="gray", linestyle=":", linewidth=1)
axes[3].set_title("|mu| médio — perto de 0 = posterior colapsado")

for ax in axes:
    ax.set_xlabel("época"); ax.legend()
plt.tight_layout(); plt.show()